In [19]:
import os
from pathlib import Path
import re
from typing import Dict, List, Set

import duckdb
from dotenv import load_dotenv
import pandas as pd

In [20]:
def infer_table_order(schema_file_path: str) -> List[str]:
    """
    Infer the order of table creation based on foreign key dependencies.

    This function reads a SQL schema file, analyzes the CREATE TABLE statements
    and their foreign key relationships, and returns a list of table names in an
    order that ensures all dependencies are met when creating the tables.

    The order starts with tables that have no dependencies (top-level tables) and
    ends with tables that depend on others.

    Args:
        schema_file_path (str): The file path to the SQL schema file.

    Returns:
        List[str]: A list of table names in the order they should be created,
                   starting with tables with no dependencies.

    Raises:
        FileNotFoundError: If the specified schema file cannot be found.
        ValueError: If no valid CREATE TABLE statements are found in the file.
    """
    # Read the schema from the file
    try:
        with open(schema_file_path, "r") as file:
            schema_sql = file.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"Schema file not found: {schema_file_path}")

    # Remove comments and split into CREATE TABLE statements
    schema_sql = remove_comments(schema_sql)
    create_statements = split_create_statements(schema_sql)

    if not create_statements:
        raise ValueError("No valid CREATE TABLE statements found in the schema file.")

    # Build dependency graph
    dependencies = build_dependency_graph(create_statements)

    # Perform topological sort
    sorted_tables = topological_sort(dependencies)

    return sorted_tables  # This is already in the desired order (top-level first)


def remove_comments(sql: str) -> str:
    """
    Remove SQL comments from the input string.

    Args:
        sql (str): The input SQL string.

    Returns:
        str: The SQL string with comments removed.
    """
    return re.sub(r"--.*$", "", sql, flags=re.MULTILINE)


def split_create_statements(sql: str) -> List[str]:
    """
    Split the SQL string into individual CREATE TABLE statements.

    Args:
        sql (str): The input SQL string.

    Returns:
        List[str]: A list of CREATE TABLE statements.
    """
    return [stmt.strip() for stmt in re.split(r"(?i)CREATE\s+OR\s+REPLACE\s+TABLE", sql)[1:]]


def build_dependency_graph(create_statements: List[str]) -> Dict[str, Set[str]]:
    """
    Build a graph of table dependencies based on foreign key relationships.

    Args:
        create_statements (List[str]): A list of CREATE TABLE statements.

    Returns:
        Dict[str, Set[str]]: A dictionary where keys are table names and values
                             are sets of tables they depend on.
    """
    dependencies: Dict[str, Set[str]] = {}

    for statement in create_statements:
        match = re.match(r"\s*(\w+)", statement)
        if match:
            current_table = match.group(1)
            dependencies[current_table] = set()

            fk_references = re.findall(r"FOREIGN KEY.*REFERENCES\s+(\w+)", statement, re.IGNORECASE)
            for referenced_table in fk_references:
                dependencies[current_table].add(referenced_table)

    return dependencies


def topological_sort(graph: Dict[str, Set[str]]) -> List[str]:
    """
    Perform a topological sort on the dependency graph.

    This implementation uses depth-first search to produce a topological ordering.
    The result is reversed to get the desired order (tables with no dependencies first).

    Args:
        graph (Dict[str, Set[str]]): The dependency graph.

    Returns:
        List[str]: A list of table names in topological order.
    """
    visited: Set[str] = set()
    result: List[str] = []

    def dfs(node: str) -> None:
        if node in visited:
            return
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor)
        result.append(node)

    for node in graph:
        if node not in visited:
            dfs(node)

    return result  # No need to reverse, this is already in the desired order

In [21]:
SCHEMA = Path("../sql/create_cb_experiment_tables_ddb.sql")

In [22]:
try:
    table_order = infer_table_order(str(SCHEMA))
    print("Inferred table order (top-level first):", table_order)
except (FileNotFoundError, ValueError) as e:
    print(f"Error: {e}")

Inferred table order (top-level first): ['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']


In [23]:
def copy_database_to_motherduck(source_db_path, md_db_name, tables=None):
    conn = duckdb.connect(":memory:")
    load_dotenv()

    md_token = os.getenv("MD_TOKEN")
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")

    conn.sql("ATTACH 'md:';")

    try:
        # Attach the source database
        print(f"ATTACH '{source_db_path}' AS local_source_db")
        conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

        print("USE 'local_source_db'")
        conn.execute("USE 'local_source_db'")

        # Get the list of tables from the source database
        if tables is None:
            tables = [table[0] for table in conn.execute("SHOW TABLES").fetchall()]

        # Create the new database
        print(f"CREATE OR REPLACE DATABASE {md_db_name}")
        conn.execute(f"CREATE OR REPLACE DATABASE {md_db_name}")

        # Copy the schema from the source database to the new database
        print(f"COPY FROM DATABASE local_source_db TO {md_db_name} (SCHEMA)")
        conn.execute(f"COPY FROM DATABASE local_source_db TO {md_db_name} (SCHEMA)")

        # Copy data for each table
        print(tables)
        for table_name in tables:
            print(f"INSERT INTO {md_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}")
            conn.execute(
                f"INSERT INTO {md_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}"
            )

        print(f"INFO: Database '{md_db_name}' created and populated successfully.")

    except Exception as e:
        print(f"ERROR: An error occurred: {str(e)}")

    finally:
        conn.close()

    return md_db_name

In [28]:
CB_DATABASE = Path("../data/Experiments_CB_2023_Q4.ddb")
CB_DATABASE_MD = str(CB_DATABASE.stem)

In [25]:
table_order

['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']

In [ ]:
md_db = copy_database_to_motherduck(CB_DATABASE, CB_DATABASE_MD, table_order)

In [26]:
def compare_databases(original_db, md_db):
    print(f"Comparing - ORIGINAL: {original_db} vs NEW: {md_db}")

    load_dotenv()
    md_token = os.getenv("MD_TOKEN")
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")

    conn_new = duckdb.connect(f"md:{md_db}?motherduck_token={md_token}")
    conn_original = duckdb.connect(original_db)

    tables_original = conn_original.execute("SHOW TABLES").fetchall()
    tables_new = conn_new.execute("SHOW TABLES").fetchall()

    if len(tables_original) != len(tables_new):
        print(f"Number of tables mismatch. Original: {len(tables_original)}, New: {len(tables_new)}")
        return False

    for table in tables_original:
        table_name = table[0]
        print(f"Comparing table: {table_name}")

        schema_original = conn_original.execute(f"DESCRIBE {table_name}").fetchall()
        schema_new = conn_new.execute(f"DESCRIBE {table_name}").fetchall()

        if schema_original != schema_new:
            print(f"Schema mismatch for table {table_name}")
            print("Original schema:")
            print(
                pd.DataFrame(
                    schema_original, columns=["Column", "Type", "Null", "Key", "Default", "Extra"]
                )
            )
            print("\nNew schema:")
            print(
                pd.DataFrame(schema_new, columns=["Column", "Type", "Null", "Key", "Default", "Extra"])
            )
            return False

        # Compare data
        data_original = conn_original.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()
        data_new = conn_new.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()

        if data_original != data_new:
            print(f"Data mismatch for table {table_name}")

            # For TrialCBs table, perform a more detailed comparison
            if table_name == "TrialCBs":
                df_original = pd.DataFrame(data_original, columns=[col[0] for col in schema_original])
                df_new = pd.DataFrame(data_new, columns=[col[0] for col in schema_new])

                # Compare row counts
                if len(df_original) != len(df_new):
                    print(f"Row count mismatch. Original: {len(df_original)}, New: {len(df_new)}")

                # Compare column values
                for column in df_original.columns:
                    if not df_original[column].equals(df_new[column]):
                        print(f"Mismatch in column: {column}")
                        print("First few mismatching rows:")
                        mismatch = df_original[column] != df_new[column]
                        print(
                            pd.concat(
                                [df_original[mismatch], df_new[mismatch]],
                                axis=1,
                                keys=["Original", "New"],
                            ).head()
                        )

            return False

        # Compare row count
        count_original = conn_original.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        count_new = conn_new.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

        if count_original != count_new:
            print(
                f"Row count mismatch for table {table_name}. Original: {count_original}, New: {count_new}"
            )
            return False

    print("All tables and data match!")
    return True

In [30]:
databases_identical = compare_databases(CB_DATABASE, CB_DATABASE_MD)
print(f"Databases are identical: {databases_identical}")

Comparing - ORIGINAL: ../data/Experiments_CB_2023_Q4.ddb vs NEW: Experiments_CB_2023_Q4
Comparing table: EventCBs
Comparing table: ExperimentCBs
Comparing table: ResponseCBs
Comparing table: TrialCBs
Data mismatch for table TrialCBs
Databases are identical: False
